# ArcFace 기반 Grad-CAM 시각화 노트북

이 노트북에서는 ArcFace 모델을 활용하여 `downloaded_datasets` 폴더 내 임의의 10장 이미지에 대해 Grad-CAM을 적용하고, 시각화 결과를 확인합니다.

---

## 1. 필요한 라이브러리 임포트

In [ ]:
import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import cv2
import glob
import random
import os
from insightface.model_zoo import get_model
from insightface.utils import face_align
from PIL import Image


## 2. ArcFace 모델 불러오기 및 준비

In [ ]:
# insightface ArcFace 모델 로드 (r100, glint360k 사전학습 가중치)
arcface_model = get_model('arcface_r100_v1')
arcface_model.prepare(ctx_id=-1)  # CPU 사용, GPU는 ctx_id=0
arcface_model.eval()


## 3. 이미지 10장 선택 및 전처리

In [ ]:
# downloaded_datasets 내 모든 하위 폴더의 이미지 경로 수집
def get_all_image_paths(root_dir, exts=("jpg", "jpeg", "png")):
    image_paths = []
    for ext in exts:
        image_paths.extend(glob.glob(os.path.join(root_dir, "**", f"*.{ext}"), recursive=True))
    return image_paths

image_root = "../downloaded_datasets" if not os.path.exists("downloaded_datasets") else "downloaded_datasets"
all_image_paths = get_all_image_paths(image_root)

# 임의의 10장 이미지 선택
sample_image_paths = random.sample(all_image_paths, 10)

# ArcFace 입력 전처리 함수 (얼굴 정렬 및 112x112 resize)
def preprocess_image(img_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (112, 112))
    img = np.transpose(img, (2, 0, 1))
    img = np.expand_dims(img, axis=0)
    img = img.astype(np.float32) / 255.0
    return torch.from_numpy(img)

preprocessed_images = [preprocess_image(p) for p in sample_image_paths]


## 4. ArcFace 모델로 예측 수행

In [ ]:
# ArcFace 모델로 특징 추출 (예측)
features = []
with torch.no_grad():
    for img in preprocessed_images:
        feat = arcface_model(img)[0].cpu().numpy()
        features.append(feat)


## 5. Grad-CAM 구현 및 시각화

In [ ]:
# ArcFace 마지막 Conv 레이어를 대상으로 Grad-CAM 구현
class GradCAM:
    def __init__(self, model, target_layer_name):
        self.model = model
        self.target_layer = dict([*model._modules.items()])[target_layer_name]
        self.gradients = None
        self.activations = None
        self.hook_handles = []
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()
        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()
        self.hook_handles.append(self.target_layer.register_forward_hook(forward_hook))
        self.hook_handles.append(self.target_layer.register_backward_hook(backward_hook))

    def __call__(self, x, index=None):
        self.model.zero_grad()
        out = self.model(x)
        if index is None:
            index = out[0].argmax().item()
        target = out[0][index]
        target.backward(retain_graph=True)
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        grad_cam_map = (weights * self.activations).sum(dim=1, keepdim=True)
        grad_cam_map = F.relu(grad_cam_map)
        grad_cam_map = F.interpolate(grad_cam_map, size=(112, 112), mode='bilinear', align_corners=False)
        grad_cam_map = grad_cam_map[0][0].cpu().numpy()
        grad_cam_map = (grad_cam_map - grad_cam_map.min()) / (grad_cam_map.max() - grad_cam_map.min() + 1e-8)
        return grad_cam_map

    def remove_hooks(self):
        for handle in self.hook_handles:
            handle.remove()

# ArcFace r100의 마지막 conv 레이어 이름은 'body.22' (insightface 구조 기준)
gradcam = GradCAM(arcface_model, target_layer_name='body.22')

# 각 이미지에 대해 Grad-CAM 맵 생성
gradcam_maps = []
for img in preprocessed_images:
    cam = gradcam(img)
    gradcam_maps.append(cam)


## 6. Grad-CAM 결과 출력

In [ ]:
# 원본 이미지와 Grad-CAM 맵을 나란히 시각화
for idx, (img_path, cam) in enumerate(zip(sample_image_paths, gradcam_maps)):
    orig = cv2.imread(img_path)
    orig = cv2.cvtColor(orig, cv2.COLOR_BGR2RGB)
    orig = cv2.resize(orig, (112, 112))
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(orig, 0.5, heatmap, 0.5, 0)
    plt.figure(figsize=(8, 3))
    plt.subplot(1, 3, 1)
    plt.imshow(orig)
    plt.title('Original')
    plt.axis('off')
    plt.subplot(1, 3, 2)
    plt.imshow(cam, cmap='jet')
    plt.title('Grad-CAM')
    plt.axis('off')
    plt.subplot(1, 3, 3)
    plt.imshow(overlay)
    plt.title('Overlay')
    plt.axis('off')
    plt.suptitle(f"Sample {idx+1}")
    plt.show()


# requirements.txt에 추가할 라이브러리 목록

아래 라이브러리들을 requirements.txt에 추가하세요.

- torch
- torchvision
- numpy
- matplotlib
- opencv-python
- insightface
- pillow